# Feature engineering - advanced data preparation pipeline  | Sebislaw

## Libraries

In [1]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
#from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier

from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd
from pandas.plotting import scatter_matrix

import matplotlib.pyplot as plt
import plotly.express as px
from pandas.plotting import parallel_coordinates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold, RandomizedSearchCV
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.neural_network import MLPClassifier

import xgboost as xgb
from xgboost import XGBClassifier
#from catboost import CatBoostClassifier
import optuna
# from tabpfn import TabPFNClassifier


import optuna
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Data

In [2]:
# Modele bazowe
lr = LogisticRegression(penalty='l1', C=0.22814815636570945, solver='saga', max_iter=3000)
rf1 = RandomForestClassifier(n_estimators=500, criterion='entropy')
rf2 = RandomForestClassifier(n_estimators=50, max_depth=5, criterion='entropy', max_features=None)

# Komitet głosujący
voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('rf1', rf1), ('rf2', rf2)],
    voting='soft'
)

In [3]:
data_path = '../data'
pd.set_option('display.max_columns', None)

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

## Data preparation pipeline

In [4]:
def prepare_data(df):
        
    """
    This function duplicates and flips a game record.
    Now two records with the same data are present, 
    but viewed from perspectives of two different teams.
    """
    
    df = df[[
         'Season', 'DayNum', 'NumOT',
         'WTeamID',  'WScore', 'WLoc',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
         'LTeamID', 'LScore',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'
    ]]
    dfswap = df[[
         'Season', 'DayNum', 'NumOT',
         'LTeamID', 'LScore', 'WLoc',
         'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF',
         'WTeamID',  'WScore',
         'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF',
    ]].copy()
    
    dfswap.loc[df['WLoc'] == 'H', 'WLoc'] = 'A'
    dfswap.loc[df['WLoc'] == 'A', 'WLoc'] = 'H'
        
    df = df.rename(columns={'WLoc': 'location'})
    dfswap = dfswap.rename(columns={'WLoc': 'location'})
        
    df.columns = [x.replace('W','T1_').replace('L','T2_') for x in list(df.columns)]
    dfswap.columns = [x.replace('L','T1_').replace('W','T2_') for x in list(dfswap.columns)]
    
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output.loc[output.location=='N','location'] = '0'
    output.loc[output.location=='H','location'] = '1'
    output.loc[output.location=='A','location'] = '-1'
    output.location = output.location.astype(int)
        
    output['PointDiff'] = output['T1_Score'] - output['T2_Score']
    
    return output

def get_data(regular_results, tourney_results, seeds, prepared=False, location_multiplier=[1, 1], win_ratio_days_back=14):

    """
    This function uses the prepare_data function in order to create
    a data frame with season statistics for each team.
    These statistics are added to records with games played in
    tournament to make data 'x' used in model to predict the game 
    result 'y'. The output is a data frame that contains data 'x'
    and also label 'y' can be easily calculated based on score difference in matches.
    """

    if prepared:
        regular_data = regular_results.copy()
        tourney_data = tourney_results.copy()
    else:
        # make data frames with extra rows to represent the perspective of losing team
        regular_data = prepare_data(regular_results)
        tourney_data = prepare_data(tourney_results)

    # ----------------------------------- Add reward/penalty for playing in home or away
    if location_multiplier[0] == 1 and location_multiplier[1] == 1:
        # data frame with mean game statistics for a given team in a given season
        season_statistics  = regular_data.groupby(["Season", 'T1_TeamID'])[
            [
                'T1_Score', 'T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA',
                'T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF',
                'T2_Score', 'T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA',
                'T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF',
                'PointDiff'
            ]
        ].agg('mean').reset_index()
    else:
        # Define which columns to adjust (you can add or remove columns as needed)
        T1_cols = ['T1_Score','T1_FGM','T1_FGA','T1_FGM3','T1_FGA3','T1_FTM','T1_FTA','T1_OR','T1_DR','T1_Ast','T1_TO','T1_Stl','T1_Blk','T1_PF']
        T2_cols = ['T2_Score','T2_FGM','T2_FGA','T2_FGM3','T2_FGA3','T2_FTM','T2_FTA','T2_OR','T2_DR','T2_Ast','T2_TO','T2_Stl','T2_Blk','T2_PF']
    
        # Convert the relevant columns to float before applying the adjustment function.
        cols_to_float = T1_cols + T2_cols
        regular_data[cols_to_float] = regular_data[cols_to_float].astype(float)
        
        def adjust_stats(row):
            # Determine multipliers based on location
            if row['location'] == 1:
                factor_T1 = location_multiplier[0]  # penalize Team1 stats (home)
                factor_T2 = location_multiplier[1]  # boost Team2 stats
            elif row['location'] == -1:
                factor_T1 = location_multiplier[1]  # boost Team1 stats (away)
                factor_T2 = location_multiplier[0]  # penalize Team2 stats
            else:
                factor_T1 = 1.0
                factor_T2 = 1.0
        
            # Adjust Team1 stats
            for col in T1_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T1
        
            # Adjust Team2 stats
            for col in T2_cols:
                if col in row and pd.notnull(row[col]):
                    row[col] = row[col] * factor_T2
        
            # Recalculate derived statistics (if needed)
            if 'T1_Score' in row and 'T2_Score' in row:
                row['PointDiff'] = row['T1_Score'] - row['T2_Score']
            return row
        
        # Apply the adjustment function row-wise.
        regular_data_adjusted = regular_data.apply(adjust_stats, axis=1)
        
        # Now group by Season and T1_TeamID to compute season averages for the adjusted statistics.
        stats_columns = T1_cols[1:] + T2_cols[1:] + ['PointDiff']  # Exclude T1_TeamID from stats if present.
        season_statistics = regular_data_adjusted.groupby(["Season", 'T1_TeamID'])[stats_columns].agg('mean').reset_index()
    # -----------------------------------
    
    # mean statistics for team and team's opponent's
    season_statistics_T1 = season_statistics.copy()
    season_statistics_T2 = season_statistics.copy()
    
    season_statistics_T1.columns = ["T1_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T1.columns)]
    season_statistics_T2.columns = ["T2_" + x.replace("T1_","").replace("T2_","opponent_") for x in list(season_statistics_T2.columns)]
    season_statistics_T1.columns.values[0] = "Season"
    season_statistics_T2.columns.values[0] = "Season"
    season_statistics_T1 = season_statistics_T1.rename(columns={'T1_Score': 'T1_Score_mean'})
    season_statistics_T2 = season_statistics_T2.rename(columns={'T2_Score': 'T2_Score_mean'})
    
#     print('season_statistics_T2 in get data')
#     print(season_statistics_T2)
    
    # data frame containing game's result
#     print('tourney_data BEFORE MERGE')
#     print(tourney_data)
    tourney_data = tourney_data[['Season', 'DayNum', 'T1_TeamID', 'T1_Score', 'T2_TeamID' ,'T2_Score', 'location']]
    tourney_data = pd.merge(tourney_data, season_statistics_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, season_statistics_T2, on = ['Season', 'T2_TeamID'], how = 'left')
#     print('tourney_data AFTER MERGE NO NA')
#     print(tourney_data[tourney_data['T1_FGM'].notna()])

    calculate_win_ratio_days_back = 132 - win_ratio_days_back
    
    # data frame with win fraction from last x days for a given team in a given season
    last14days_stats_T1 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T1['win'] = np.where(last14days_stats_T1['PointDiff']>0,1,0)
    last14days_stats_T1 = last14days_stats_T1.groupby(['Season','T1_TeamID'])['win'].mean().reset_index(name='T1_win_ratio_14d')
    
    last14days_stats_T2 = regular_data.loc[regular_data.DayNum>calculate_win_ratio_days_back].reset_index(drop=True)
    last14days_stats_T2['win'] = np.where(last14days_stats_T2['PointDiff']<0,1,0)
    last14days_stats_T2 = last14days_stats_T2.groupby(['Season','T2_TeamID'])['win'].mean().reset_index(name='T2_win_ratio_14d')
    
    # add to tourney_data column with win fraction for winning and losing team
    tourney_data = pd.merge(tourney_data, last14days_stats_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, last14days_stats_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # get seeds with no regional division
    seeds['seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))
    
    # give each team a raw seed
    seeds_T1 = seeds[['Season','TeamID','seed']].copy()
    seeds_T2 = seeds[['Season','TeamID','seed']].copy()
    seeds_T1.columns = ['Season','T1_TeamID','T1_seed']
    seeds_T2.columns = ['Season','T2_TeamID','T2_seed']
    
    # add seeds to turney data for team 1 and team 2
    tourney_data = pd.merge(tourney_data, seeds_T1, on = ['Season', 'T1_TeamID'], how = 'left')
    tourney_data = pd.merge(tourney_data, seeds_T2, on = ['Season', 'T2_TeamID'], how = 'left')
    
    # add a seed difference column
    tourney_data["Seed_diff"] = tourney_data["T1_seed"] - tourney_data["T2_seed"]

    return tourney_data

def get_df(seeds,
            season_games, season_range, days_back,
            tourney_games, tourney_range, 
            location_multiplier=[1, 1],
           win_ratio_days_back=14):

    """
    This function uses get_data function to get a data
    frame which is then used to make 'x' and 'y' data
    used in models.
    """
    
    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]
    
    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Get final data frame
    df = get_data(regular_results, tourney_results, seeds, location_multiplier=[1, 1], win_ratio_days_back=win_ratio_days_back)
    
    return df

def get_final_df(seeds,
                   season_games, season_range, days_back,
                   tourney_games, tourney_range, 
                   SampleSubmissionStage2,
                  location_multiplier=[1, 1],
                win_ratio_days_back=14):

    """
    This function works the same as function get_x_y,
    but also creates (at the moment it's the same as get_x_y)
    aditional data points mostly with NaN values that matches
    the submission format (parsed team matchups with the sample submission file).
    """

    # Get from regular season games from specified season range and days back
    # The results of those games will be used as team statictics (additional team information for tourney games)
    regular_results = season_games[season_games['Season'].isin(season_range)]
    regular_results = regular_results[regular_results['DayNum'] > 134 - days_back]

    # Get tourney games from a specified season
    # The results of those games will be used as labels
    tourney_results = tourney_games[tourney_games['Season'].isin(tourney_range)]
    
    # Assume sample_submission is a DataFrame with an "ID" column like "2023_1101_1102"
    # and tourney_results is a DataFrame with columns including: Season, WTeamID, LTeamID, DayNum, WScore, LScore, WLoc, etc.
    # Filter rows where the ID starts with the specified season (followed by an underscore)
    final_season = tourney_range[0]
    sample_submission_copy = SampleSubmissionStage2.copy()
    sample_submission = sample_submission_copy[sample_submission_copy['ID'].str.startswith(f"{final_season}_")]
    sample_submission = sample_submission.drop(columns=['Pred'])
    
    sample_submission[['Season', 'Team1', 'Team2']] = sample_submission['ID'].str.split('_', expand=True)
    sample_submission['Season'] = sample_submission['Season'].astype(float)
    sample_submission['Team1'] = sample_submission['Team1'].astype(float)
    sample_submission['Team2'] = sample_submission['Team2'].astype(float)
    
    regular_data_final = prepare_data(regular_results)
    tourney_data_final  = prepare_data(tourney_results)
    
#     print("-------------")
#     print(regular_data_final)
    
#     print(list(tourney_data_final))
    sample_submission = sample_submission.rename(columns={'Team1': 'T1_TeamID', 'Team2': 'T2_TeamID'})
#     print('list(sample_submission)')
#     print(list(sample_submission))
    tourney_data_final = pd.merge(
        sample_submission,
        tourney_data_final,
        on=['Season', 'T1_TeamID', 'T2_TeamID'],
#         left_on=['Season', 'Team1', 'Team2'],
#         right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
#     tourney_data_final.rename(columns={'Team1': 'T1_TeamID', 'Team2': 'T2_TeamID'}, inplace=True)
#     print("TESM AIN")
#     print(tourney_data_final['T1_TeamID'])
    tourney_data_final = tourney_data_final[['Season', 'DayNum', 'NumOT', 'T1_TeamID', 'T1_Score', 'location', 'T1_FGM', 'T1_FGA', 'T1_FGM3', 'T1_FGA3', 'T1_FTM', 'T1_FTA', 'T1_OR', 'T1_DR', 'T1_Ast', 'T1_TO', 'T1_Stl', 'T1_Blk', 'T1_PF', 'T2_TeamID', 'T2_Score', 'T2_FGM', 'T2_FGA', 'T2_FGM3', 'T2_FGA3', 'T2_FTM', 'T2_FTA', 'T2_OR', 'T2_DR', 'T2_Ast', 'T2_TO', 'T2_Stl', 'T2_Blk', 'T2_PF', 'PointDiff']]
#     print(tourney_data_final['T1_TeamID'])
#     print("-------------")
    
    tourney_data =  get_data(regular_data_final, tourney_data_final,
                             seeds, prepared=True,
                          win_ratio_days_back=win_ratio_days_back)
#     print("NOTNA")
#     print(tourney_data_final['T1_TeamID'])
#     print(tourney_data[tourney_data['T1_FGM'].notna()])
    tourney_data = pd.merge(
        sample_submission,
        tourney_data,
        on=['Season', 'T1_TeamID', 'T2_TeamID'],
#         left_on=['Season', 'Team1', 'Team2'],
#         right_on=['Season', 'T1_TeamID', 'T2_TeamID'],
        how='left'
    )
#     tourney_data['T1_TeamID'] = tourney_data['Team1']
#     tourney_data['T2_TeamID'] = tourney_data['Team2']
#     tourney_data = tourney_data.drop(['ID', 'Team1', 'Team2'], axis=1)
    tourney_data = tourney_data.drop(['ID'], axis=1)
    
    return tourney_data

def correct_predictions_based_on_seed(x, y, maximum_favoured_seed = 4, number_of_added_columns=1):
    """
    Sets the winnning chance to 1 or 0 based on the seed difference.
    """
    for i in range(len(x)):
        if x[i][-1-number_of_added_columns] <= (-16 + maximum_favoured_seed * 2 - 1):
            y[i] = 1
        elif x[i][-1-number_of_added_columns] >= (16 - maximum_favoured_seed * 2 + 1):
            y[i] = 0
    return y

def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[7:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

def get_all_core_data(
    regular_results, tourney_results, seeds, SampleSubmissionStage2,
    final_season = 2024, # the season we want to predict, so for out submission it will be 2025
    start_season = 2005, # from which ponit should we begin creating data
    season_years_list  = [[i-1, i] for i in range(2005, 2024+1)], # at which seasons to look at when calculating team's stats
    days_back = 15, # how many days back from the start of tourney to calculate team's stats per season
    maximum_favoured_seed = 4, # Set the predicted probability of winning to 1 for seeds <= maximum_favoured_seed and to 0 for >= 16-maximum_favoured_seed
    location_multiplier=[0.95, 1.05], # home penalty, away bonus 
    include_men = True, # include M... data sets when preparing x and y
    include_women = True, # include W... data sets when preparing x and y
    win_ratio_days_back = 14):

    """
    The idea of this function is to easily get data needed to train and test the model later on, with minimal code
    to not clutter the netebook.
    
    This function outputs df, x, y, df_final, x_final_season, y_final_season.
    
    df is a data frame with first 6 columns from tourney games and other calculated from other data frames.
    
    Adding a column to df and executing x_y_from_data_frame(df) function will yield x with added data.
    
    Note that df_final has the same structure as df, but also with rows with NaNs. The rows with missing information are there
    to match the sumbission file format. The separation of those data frames is to ensure that information from last season doesn't
    leak into training data due to poorly written code.
    
    x has df columns from location onwardsthe columns before that are from tourney games and are used to calculate y (based on points).
    
    y has label 0 or 1 (lose or win) and nan if the correspoinding data in x was nan.
    
    There is also x_final_season and y_final_season aquired from df_final, which are the same as x and y, but like df_final, they have NaNs. 
    """

    # This ensures that we simulate the scenario in competition
    tourney_results_final = tourney_results[tourney_results['Season'] == final_season]
    tourney_results = tourney_results[tourney_results['Season'] < final_season]
    regular_results = regular_results[regular_results['Season'] != 2020] # This year had no tournament data
    tourney_years_list = [[i] for i in range(start_season, final_season+1)]
    
    # Arrays to store data
    data = []
    data_final = []
    for season_years, tourney_years in zip(season_years_list, tourney_years_list):
            
        # Create data separately for the of games
        # The separation is to ensure there is no data leak
        if tourney_years[0] == final_season:
            data_tmp = get_final_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results_final, tourney_years,
                SampleSubmissionStage2=SampleSubmissionStage2,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data_final.append(data_tmp)
        else:
            data_tmp = get_df(
                seeds,
                regular_results, season_years, days_back,
                tourney_results, tourney_years,
                location_multiplier=location_multiplier,
                win_ratio_days_back=win_ratio_days_back
            )
            data.append(data_tmp)
            
    df = pd.concat(data, ignore_index=True)
    df_final = pd.concat(data_final, ignore_index=True)
    
    return df, df_final

def brier_for_all_years(year_range, season_years_list):
    
    df_train_women_list = []
    df_test_women_list = []
    df_train_men_list = []
    df_test_men_list = []
    
    for year, season_years in zip(year_range, season_years_list):
    
        final_season = year

        for sex in ['woman', 'man']:
            
            if sex == 'woman':
                include_men = False
                include_women = True
            elif sex == 'man':
                include_men = True
                include_women = False

            # ----------------------------------------------------------
            # READ DATA
            regular_results = pd.concat([
                MRegularSeasonDetailedResults.copy() if include_men else None,
                WRegularSeasonDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            tourney_results = pd.concat([
                MNCAATourneyDetailedResults.copy() if include_men else None,
                WNCAATourneyDetailedResults.copy() if include_women else None
            ], ignore_index=True)
            seeds = pd.concat([
                MNCAATourneySeeds.copy() if include_men else None,
                WNCAATourneySeeds.copy() if include_women else None
            ], ignore_index=True)
            # ----------------------------------------------------------
            # GET ALL DATA NEEDED TO USE THE MODELS
            df_train, df_test = get_all_core_data(
                regular_results, tourney_results, seeds, SampleSubmissionStage2, final_season = final_season,
                start_season = start_season, season_years_list  = season_years, days_back = days_back,
                maximum_favoured_seed = maximum_favoured_seed, location_multiplier=location_multiplier,
                include_men = include_men, include_women = include_women, win_ratio_days_back = win_ratio_days_back)
            # ----------------------------------------------------------
            # ADD TEAM AND COACH ELO AND REPLACE NAN WITH MEAN
            def add_elo_columns_team1(df):
                df = df.copy()
                df = pd.merge(
                    df,
                    elo[['Season', 'TeamID', 'TeamELO', 'CoachELO']],
                    left_on=['Season', 'T1_TeamID'],
                    right_on=['Season', 'TeamID'],
                    how='left'
                )
                df = df.drop(['TeamID'], axis=1)
                df = df.rename(columns={'TeamELO': 'T1_TeamELO', 'CoachELO': 'T1_CoachELO'})
                return df

            def add_elo_columns_team2(df):
                df = df.copy()
                df = pd.merge(
                    df,
                    elo[['Season', 'TeamID', 'TeamELO', 'CoachELO']],
                    left_on=['Season', 'T2_TeamID'],
                    right_on=['Season', 'TeamID'],
                    how='left'
                )
                df = df.drop(['TeamID'], axis=1)
                df = df.rename(columns={'TeamELO': 'T2_TeamELO', 'CoachELO': 'T2_CoachELO'})
                return df
            df_train = add_elo_columns_team1(df_train)
            df_train = add_elo_columns_team2(df_train)
            df_test  = add_elo_columns_team1(df_test)
            df_test  = add_elo_columns_team2(df_test)
            # ----------------------------------------------------------
            if sex == 'woman':
                df_train_women_list.append(df_train.copy())
                df_test_women_list.append(df_test.copy())
            elif sex == 'man':
                df_train_men_list.append(df_train.copy())
                df_test_men_list.append(df_test.copy())
                
        for i in range(len(df_train_men_list)):
            df_train_women_list[i] = df_train_women_list[i][list(df_train_men_list[0])]
            df_test_women_list[i] = df_test_women_list[i][list(df_train_men_list[0])]
            
    return df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list

In [5]:
final_elo_team = pd.read_csv(join(data_path, 'final_elo_team.csv'))
final_elo_team = final_elo_team.sort_values(by="TeamID").reset_index(drop=True)
final_elo_team

,TeamID,TeamELO
0,1101,1459.186427
1,1102,1285.078651
2,1103,1727.740773
3,1104,2069.064662
4,1105,1025.906452
...,...,...
744,3476,1187.241756
745,3477,1113.947230
746,3478,1178.688631
747,3479,1190.234158


In [6]:
final_elo_coach = pd.read_csv(join(data_path, 'final_elo_coach.csv'))
final_elo_coach = final_elo_coach.sort_values(by="TeamID").reset_index(drop=True)
final_elo_coach

,Season,CoachName,CoachELO,TeamID
0,2025.0,brette_tanner,1557.695414,1101.0
1,2025.0,joe_scott,1388.153462,1102.0
2,2025.0,john_groce,1856.454628,1103.0
3,2025.0,nate_oats,2129.457038,1104.0
4,2025.0,otis_hughley_jr,1271.731597,1105.0
...,...,...,...,...
359,2025.0,chris_kraus,1334.420774,1476.0
360,2025.0,jaret_von_rosenberg,1292.013185,1477.0
361,2025.0,nate_champion,1291.677335,1478.0
362,2025.0,gary_manchel,1448.532417,1479.0


In [7]:
elo = pd.read_csv(join(data_path, 'elo.csv'))
elo['CoachELO'] = elo['CoachELO'].fillna(elo['CoachELO'].mean())
elo = elo.drop(['CoachName'], axis=1)
elo = elo.loc[elo.groupby(['Season', 'TeamID'])['DayNum'].idxmax()]
elo

,Season,DayNum,TeamID,TeamELO,CoachELO
7369,1985,127,1102,1384.975555,1386.121212
6759,1985,119,1103,1456.386146,1458.732587
7735,1985,144,1104,1662.447995,1663.573415
7284,1985,126,1106,1435.719069,1435.498557
7609,1985,131,1108,1642.627949,1643.394432
...,...,...,...,...,...
676019,2025,132,3476,1192.740066,1612.184052
674469,2025,121,3477,1126.698632,1612.184052
675811,2025,129,3478,1195.597965,1612.184052
674731,2025,122,3479,1213.173465,1612.184052


In [8]:
import pandas as pd

# Assume elo, final_elo_team, and final_elo_coach are already loaded

# Step 1: Create a copy of only the 2025 rows
elo_2025 = elo[elo['Season'] == 2025].copy()

# Step 2: Merge new TeamELO values (only for 2025 teams)
elo_2025 = elo_2025.merge(final_elo_team, on="TeamID", how="left", suffixes=("", "_new"))
elo_2025["TeamELO"] = elo_2025["TeamELO_new"].combine_first(elo_2025["TeamELO"])
elo_2025.drop(columns=["TeamELO_new"], inplace=True)

# Step 3: Merge new CoachELO values (only for 2025 teams)
final_elo_coach_sub = final_elo_coach[['TeamID', 'CoachELO']]
elo_2025 = elo_2025.merge(final_elo_coach_sub, on="TeamID", how="left", suffixes=("", "_new"))
elo_2025["CoachELO"] = elo_2025["CoachELO_new"].combine_first(elo_2025["CoachELO"])
elo_2025.drop(columns=["CoachELO_new"], inplace=True)

# Step 4: Replace ONLY the 2025 rows in the original DataFrame
elo_updated = pd.concat([elo[elo["Season"] != 2025], elo_2025], ignore_index=True)

# Step 5: Ensure correct order (optional)
elo_updated = elo_updated.sort_values(by=["Season", "TeamID"]).reset_index(drop=True)

elo_updated


,Season,DayNum,TeamID,TeamELO,CoachELO
0,1985,127,1102,1384.975555,1386.121212
1,1985,119,1103,1456.386146,1458.732587
2,1985,144,1104,1662.447995,1663.573415
3,1985,126,1106,1435.719069,1435.498557
4,1985,131,1108,1642.627949,1643.394432
...,...,...,...,...,...
22871,2025,132,3476,1187.241756,1612.184052
22872,2025,121,3477,1113.947230,1612.184052
22873,2025,129,3478,1178.688631,1612.184052
22874,2025,122,3479,1190.234158,1612.184052


In [9]:
elo = elo_updated.copy()
elo

,Season,DayNum,TeamID,TeamELO,CoachELO
0,1985,127,1102,1384.975555,1386.121212
1,1985,119,1103,1456.386146,1458.732587
2,1985,144,1104,1662.447995,1663.573415
3,1985,126,1106,1435.719069,1435.498557
4,1985,131,1108,1642.627949,1643.394432
...,...,...,...,...,...
22871,2025,132,3476,1187.241756,1612.184052
22872,2025,121,3477,1113.947230,1612.184052
22873,2025,129,3478,1178.688631,1612.184052
22874,2025,122,3479,1190.234158,1612.184052


## Get data frame

In [10]:
columns_to_include_women = [
 'Season',
#  'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
#  'T1_Score_mean',
 'T1_FGM',
 'T1_FGA',
 'T1_FGM3',
 'T1_FGA3',
#  'T1_FTM',
#  'T1_FTA',
 'T1_OR',
#  'T1_DR',
 'T1_Ast',
 'T1_TO',
 'T1_Stl',
#  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
 'T1_opponent_FGA',
 'T1_opponent_FGM3',
 'T1_opponent_FGA3',
#  'T1_opponent_FTM',
#  'T1_opponent_FTA',
 'T1_opponent_OR',
#  'T1_opponent_DR',
 'T1_opponent_Ast',
 'T1_opponent_TO',
 'T1_opponent_Stl',
#  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
#  'T2_Score_mean',
 'T2_FGM',
 'T2_FGA',
 'T2_FGM3',
 'T2_FGA3',
#  'T2_FTM',
#  'T2_FTA',
 'T2_OR',
#  'T2_DR',
 'T2_Ast',
 'T2_TO',
 'T2_Stl',
#  'T2_Blk',
 'T2_PF',
                      
#  'T2_opponent_Score',
 'T2_opponent_FGM',
 'T2_opponent_FGA',
 'T2_opponent_FGM3',
 'T2_opponent_FGA3',
#  'T2_opponent_FTM',
#  'T2_opponent_FTA',
 'T2_opponent_OR',
#  'T2_opponent_DR',
 'T2_opponent_Ast',
 'T2_opponent_TO',
 'T2_opponent_Stl',
#  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'T1_TeamELO',
'T2_TeamELO',
    'T1_CoachELO',
    'T2_CoachELO'
]


columns_to_include_men = columns_to_include_women



year_range = [2025]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 25 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.95, 1.05] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)

In [11]:
#df_train_women_list  # turnieje

In [12]:
#df_train_men_list

In [13]:
#df_test_women_list

In [14]:
#df_test_men_list

## Save x

In [15]:
MenTest = df_test_men_list.copy()
MenTrain = df_train_men_list.copy()

WomenTest = df_test_women_list.copy()
WomenTrain = df_train_women_list.copy()

# Testowanie

In [16]:
data_path = ''
pd.set_option('display.max_columns', None)

SampleSubmissionStage2 = pd.read_csv(join('../data', 'SampleSubmissionStage2.csv'))

MenTest = df_test_men_list
MenTrain = df_train_men_list

WomenTest = df_test_women_list
WomenTrain = df_train_women_list

# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

In [17]:
def clear_na_from_x_y(x, y):
    """
    The data frame for final season is in format matching the submission file.
    This function clears NaNs from data.
    """
    # Create masks for training data:
    mask_train = ~np.isnan(x).any(axis=1) & ~np.isnan(y)
    x_clean = x[mask_train]
    y_clean = y[mask_train]
    return x_clean, y_clean

def x_y_from_data_frame(df):
    # Prepare data and labels
    x = df[list(df.columns[6:])].values
    y = np.where(
        df[['T1_Score', 'T2_Score']].isnull().any(axis=1),
        np.nan,
        np.where(df['T1_Score'] - df['T2_Score'] > 0, 1, 0)
    )
    return x, y

In [18]:
THE_LAST_YEAR = 2024
# I will be using train and test from train dataset
x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])

# Pominąć dół

In [19]:
# Define an objective function that uses StratifiedKFold CV.
def objective_lr_cv(trial, X, y, n_folds=5):  # Increased folds for better generalization
    # Stronger regularization: Reduce range of C to prevent overfitting
    C = trial.suggest_float("C", 1e-4, 10, log=True)  # Lower upper bound
    
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver_all = "liblinear" if penalty == "l1" else "lbfgs"
    solver = solver_all  # More stable for both L1 and L2

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(penalty=penalty, C=C, solver=solver, max_iter=3000))  # Increased max_iter
        ])

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict_proba(X_val_cv)[:, 1]
        scores.append(brier_score_loss(y_val_cv, y_pred))

    return np.mean(scores)

###########################
# For all
###########################

x_train_all = np.concatenate((x_train_women, x_train_men))
y_train_all = np.concatenate((y_train_women, y_train_men))

x_test_all = np.concatenate((x_test_women, x_test_men))
y_test_all = np.concatenate((y_test_women, y_test_men))

###########################
study_all = optuna.create_study(direction="minimize")
study_all.optimize(lambda trial: objective_lr_cv(trial, x_train_all, y_train_all), n_trials=100)  # Increased trials for better tuning

best_params_all = study_all.best_params
print("Best Logistic Regression CV Params (all):", best_params_all)

best_lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_all["penalty"],
                                       C=best_params_all["C"],
                                       solver="saga",
                                       max_iter=3000))
])
best_lr_all.fit(x_train_all, y_train_all)
y_pred_all_lr = best_lr_all.predict_proba(x_test_all)[:, 1]
brier_all_lr = brier_score_loss(y_test_all, y_pred_all_lr)
print("Final Logistic Regression Brier Score (all):", brier_all_lr)

[I 2025-05-25 14:24:51,641] A new study created in memory with name: no-name-5dc83821-0c57-40ba-b9d0-b08c299f031b
[I 2025-05-25 14:24:51,710] Trial 0 finished with value: 0.15800855205915326 and parameters: {'C': 0.00935200723707318, 'penalty': 'l1'}. Best is trial 0 with value: 0.15800855205915326.
[I 2025-05-25 14:24:51,819] Trial 1 finished with value: 0.15464594411411792 and parameters: {'C': 0.014910082386820497, 'penalty': 'l1'}. Best is trial 1 with value: 0.15464594411411792.
[I 2025-05-25 14:24:52,092] Trial 2 finished with value: 0.14885073474174076 and parameters: {'C': 0.5629868478021036, 'penalty': 'l1'}. Best is trial 2 with value: 0.14885073474174076.
[I 2025-05-25 14:24:52,138] Trial 3 finished with value: 0.22630007267253127 and parameters: {'C': 0.0017446962681664175, 'penalty': 'l1'}. Best is trial 2 with value: 0.14885073474174076.
[I 2025-05-25 14:24:52,305] Trial 4 finished with value: 0.1786864461067923 and parameters: {'C': 0.0009057552898334934, 'penalty': 'l2'

Best Logistic Regression CV Params (all): {'C': 0.23664383186227303, 'penalty': 'l1'}
Final Logistic Regression Brier Score (all): 0.1474332027000405


# Pominąć dół

In [20]:
year_range = [i for i in range(2011, 2020)] + [i for i in range(2022, 2025)]
brier_all_list = []

for i in range(len(year_range)):
    
    THE_LAST_YEAR = year_range[i]
    # For data leak validation
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season']))
#     print(len(MenTrain[MenTrain['Season']==THE_LAST_YEAR]))
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].head())
#     print(MenTrain[MenTrain['Season']<THE_LAST_YEAR]['Season'].tail())
#     print(MenTrain[MenTrain['Season']==THE_LAST_YEAR]['Season'])
    print("Predictions for season ", year_range[i])
    
    x_train_men, y_train_men = x_y_from_data_frame(MenTrain[MenTrain['Season']<THE_LAST_YEAR])
    x_train_women, y_train_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']<THE_LAST_YEAR])
    x_test_men, y_test_men = x_y_from_data_frame(MenTrain[MenTrain['Season']==THE_LAST_YEAR])
    x_test_women, y_test_women = x_y_from_data_frame(WomenTrain[WomenTrain['Season']==THE_LAST_YEAR])
    
    x_train_all = np.concatenate((x_train_women, x_train_men))
    y_train_all = np.concatenate((y_train_women, y_train_men))
    x_test_all = np.concatenate((x_test_women, x_test_men))
    y_test_all = np.concatenate((y_test_women, y_test_men))
    ## MODELE INNE
    best_lr_all = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", voting_clf)
    ])
    ## MODELE INNE
    
    best_lr_all.fit(x_train_all, y_train_all)
    y_prob_all = best_lr_all.predict_proba(x_test_all)[:, 1]

    score = brier_score_loss(y_test_all, y_prob_all)
    brier_all_list.append(score)
    print('LogisticRegressionCV for All trained on All', score)
    # ----------------------------------------------
print()
print('The mean score when trained on All and tested on All was: ', np.mean(brier_all_list), 
      'with a std of ', np.std(brier_all_list))

# Best params {'C': 0.23059755790690561, 'penalty': 'l1'}

Predictions for season  2011
LogisticRegressionCV for All trained on All 0.16670117935805448
Predictions for season  2012
LogisticRegressionCV for All trained on All 0.14548335769640716
Predictions for season  2013
LogisticRegressionCV for All trained on All 0.16264849589397004
Predictions for season  2014
LogisticRegressionCV for All trained on All 0.1564965471993364
Predictions for season  2015
LogisticRegressionCV for All trained on All 0.12528395084309457
Predictions for season  2016
LogisticRegressionCV for All trained on All 0.15900806529485662
Predictions for season  2017
LogisticRegressionCV for All trained on All 0.13616822579905744
Predictions for season  2018
LogisticRegressionCV for All trained on All 0.16224346889157107
Predictions for season  2019
LogisticRegressionCV for All trained on All 0.1394648949754998
Predictions for season  2022
LogisticRegressionCV for All trained on All 0.1556781150145397
Predictions for season  2023
LogisticRegressionCV for All trained on All 

In [21]:
# poprawka MenTest i WomenTest

finalowa = pd.read_csv("data/finalsolution2025.csv")
finalowa.insert(0, "Season", 2025)

finalowa.rename(columns={
    "ID1": "T1_TeamID",
    "ID2": "T2_TeamID"
}, inplace=True)

finalowa = finalowa[["Season", "T1_TeamID", "T2_TeamID", "Pred"]]




MenTest_updated = MenTest.copy()

for _, row in finalowa.iterrows():
    t1 = row['T1_TeamID']
    t2 = row['T2_TeamID']
    pred = row['Pred']
    
    mask = (MenTest_updated['T1_TeamID'] == t1) & (MenTest_updated['T2_TeamID'] == t2)
    
    if pred == 0:
        MenTest_updated.loc[mask, 'T1_Score'] = 0
        MenTest_updated.loc[mask, 'T2_Score'] = 1
    elif pred == 1:
        MenTest_updated.loc[mask, 'T1_Score'] = 1
        MenTest_updated.loc[mask, 'T2_Score'] = 0

MenTest = MenTest_updated




WomenTest_updated = WomenTest.copy()

for _, row in finalowa.iterrows():
    t1 = row['T1_TeamID']
    t2 = row['T2_TeamID']
    pred = row['Pred']
    
    mask = (WomenTest_updated['T1_TeamID'] == t1) & (WomenTest_updated['T2_TeamID'] == t2)
    
    if pred == 0:
        WomenTest_updated.loc[mask, 'T1_Score'] = 0
        WomenTest_updated.loc[mask, 'T2_Score'] = 1
    elif pred == 1:
        WomenTest_updated.loc[mask, 'T1_Score'] = 1
        WomenTest_updated.loc[mask, 'T2_Score'] = 0

WomenTest = WomenTest_updated

In [22]:
x_men_fit, y_men_fit = x_y_from_data_frame(MenTrain[MenTrain['Season'] == 2024])
x_women_fit, y_women_fit = x_y_from_data_frame(WomenTrain[WomenTrain['Season'] == 2024])

x_men, y_men = x_y_from_data_frame(MenTest[MenTest['Season'] == 2025])
x_women, y_women = x_y_from_data_frame(WomenTest[WomenTest['Season'] == 2025])


mask = ~np.isnan(x_men).any(axis=1) & ~np.isnan(y_men)
x_men = x_men[mask]
y_men = y_men[mask]

mask = ~np.isnan(x_women).any(axis=1) & ~np.isnan(y_women)
x_women = x_women[mask]
y_women = y_women[mask]

best_lr_all.fit(x_men_fit, y_men_fit)
y_prob_men = best_lr_all.predict_proba(x_men)[:, 1]
men_score = brier_score_loss(y_men, y_prob_men)

best_lr_all.fit(x_women_fit, y_women_fit)
y_prob_women = best_lr_all.predict_proba(x_women)[:, 1]
women_score = brier_score_loss(y_women, y_prob_women)


print("Predictions for season ", 2025)


print('LogisticRegressionCV for men trained on men', men_score)
print('LogisticRegressionCV for women trained on women', women_score)
print('LogisticRegressionCV for ALL trained on ALL', (men_score + women_score)/2)

Predictions for season  2025
LogisticRegressionCV for men trained on men 0.15449732702589689
LogisticRegressionCV for women trained on women 0.10479537007186847
LogisticRegressionCV for ALL trained on ALL 0.12964634854888268


# Od dołu

In [23]:
columns_to_include_men = [
 'Season',
#  'DayNum',
 'T1_TeamID',
 'T1_Score',
 'T2_TeamID',
 'T2_Score',
 'location',
                      
  'T1_Score_mean',
 'T1_FGM',
# 'T1_FGA',
 'T1_FGM3',
# 'T1_FGA3',
  'T1_FTM',
  'T1_FTA',
 'T1_OR',
  'T1_DR',
# 'T1_Ast',
# 'T1_TO',
# 'T1_Stl',
  'T1_Blk',
 'T1_PF',
                      
#  'T1_opponent_Score',
 'T1_opponent_FGM',
# 'T1_opponent_FGA',
 'T1_opponent_FGM3',
# 'T1_opponent_FGA3',
  'T1_opponent_FTM',
 'T1_opponent_FTA',
 'T1_opponent_OR',
  'T1_opponent_DR',
# 'T1_opponent_Ast',
# 'T1_opponent_TO',
# 'T1_opponent_Stl',
  'T1_opponent_Blk',
 'T1_opponent_PF',
 'T1_PointDiff',
                      
  'T2_Score_mean',
 'T2_FGM',
# 'T2_FGA',
 'T2_FGM3',
# 'T2_FGA3',
  'T2_FTM',
  'T2_FTA',
 'T2_OR',
  'T2_DR',
# 'T2_Ast',
# 'T2_TO',
# 'T2_Stl',
  'T2_Blk',
 'T2_PF',
                      
  'T2_opponent_Score',
 'T2_opponent_FGM',
# 'T2_opponent_FGA',
 'T2_opponent_FGM3',
# 'T2_opponent_FGA3',
  'T2_opponent_FTM',
 'T2_opponent_FTA',
 'T2_opponent_OR',
  'T2_opponent_DR',
# 'T2_opponent_Ast',
# 'T2_opponent_TO',
# 'T2_opponent_Stl',
  'T2_opponent_Blk',
 'T2_opponent_PF',
 'T2_PointDiff',
                      
 'T1_win_ratio_14d',
 'T2_win_ratio_14d',
 'T1_seed',
 'T2_seed',
 'Seed_diff',
 'T1_TeamELO',
'T2_TeamELO',
    'T1_CoachELO',
    'T2_CoachELO'
]


columns_to_include_women = columns_to_include_men



year_range = [2025]
start_season = 2010 # from which ponit should we begin creating data
season_years_list  = [[[i] for i in range(start_season, year+1)] for year in year_range] # at which seasons to look at when calculating team's stats
days_back = 14 # how many days back from the start of tourney to calculate team's stats per season
location_multiplier = [0.5, 1.5] # home penalty, away bonus ex.
win_ratio_days_back = 14 # how many days back from the tourney do we calculate win ratio
maximum_favoured_seed = 0
# -------------------------------------------
df_train_women_list, df_test_women_list, df_train_men_list, df_test_men_list = brier_for_all_years(year_range, season_years_list)
x_train_women_list = []
x_test_women_list = []
x_train_men_list = []
x_test_men_list = []
y_train_women_list = []
y_test_women_list = []
y_train_men_list = []
y_test_men_list = []
for i in range(len(year_range)):

    if len(year_range) > 1:
        df_train_women_list[i] = df_train_women_list[i][columns_to_include_women]
        df_test_women_list[i] = df_test_women_list[i][columns_to_include_women]
        df_train_men_list[i] = df_train_men_list[i][columns_to_include_men]
        df_test_men_list[i] = df_test_men_list[i][columns_to_include_men]
        
        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list[i])
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list[i])
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list[i])
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list[i])
        
        x_train_women_list.append(x_train_women)
        x_test_women_list.append(x_test_women)
        x_train_men_list.append(x_train_men)
        x_test_men_list.append(x_test_men)

        y_train_women_list.append(y_train_women)
        y_test_women_list.append(y_test_women)
        y_train_men_list.append(y_train_men)
        y_test_men_list.append(y_test_men)
    
    else:
        df_train_women_list = df_train_women_list[0][columns_to_include_women]
        df_test_women_list = df_test_women_list[0][columns_to_include_women]
        df_train_men_list = df_train_men_list[0][columns_to_include_men]
        df_test_men_list = df_test_men_list[0][columns_to_include_men]

        x_train_women, y_train_women = x_y_from_data_frame(df_train_women_list)
        x_test_women, y_test_women = x_y_from_data_frame(df_test_women_list)
        x_train_men, y_train_men = x_y_from_data_frame(df_train_men_list)
        x_test_men, y_test_men = x_y_from_data_frame(df_test_men_list)



MenTest = df_test_men_list.copy()
MenTrain = df_train_men_list.copy()

WomenTest = df_test_women_list.copy()
WomenTrain = df_train_women_list.copy()

# poprawka MenTest i WomenTest

finalowa = pd.read_csv("data/finalsolution2025.csv")
finalowa.insert(0, "Season", 2025)

finalowa.rename(columns={
    "ID1": "T1_TeamID",
    "ID2": "T2_TeamID"
}, inplace=True)

finalowa = finalowa[["Season", "T1_TeamID", "T2_TeamID", "Pred"]]




MenTest_updated = MenTest.copy()

for _, row in finalowa.iterrows():
    t1 = row['T1_TeamID']
    t2 = row['T2_TeamID']
    pred = row['Pred']
    
    mask = (MenTest_updated['T1_TeamID'] == t1) & (MenTest_updated['T2_TeamID'] == t2)
    
    if pred == 0:
        MenTest_updated.loc[mask, 'T1_Score'] = 0
        MenTest_updated.loc[mask, 'T2_Score'] = 1
    elif pred == 1:
        MenTest_updated.loc[mask, 'T1_Score'] = 1
        MenTest_updated.loc[mask, 'T2_Score'] = 0

MenTest = MenTest_updated




WomenTest_updated = WomenTest.copy()

for _, row in finalowa.iterrows():
    t1 = row['T1_TeamID']
    t2 = row['T2_TeamID']
    pred = row['Pred']
    
    mask = (WomenTest_updated['T1_TeamID'] == t1) & (WomenTest_updated['T2_TeamID'] == t2)
    
    if pred == 0:
        WomenTest_updated.loc[mask, 'T1_Score'] = 0
        WomenTest_updated.loc[mask, 'T2_Score'] = 1
    elif pred == 1:
        WomenTest_updated.loc[mask, 'T1_Score'] = 1
        WomenTest_updated.loc[mask, 'T2_Score'] = 0

WomenTest = WomenTest_updated



best_lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(penalty=best_params_all["penalty"],
                                       C=best_params_all["C"],
                                       solver="saga",
                                       max_iter=3000))
])

### MODELE INNE
best_lr_all = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", voting_clf)
])
### MODELE INNE

x_men_fit, y_men_fit = x_y_from_data_frame(MenTrain[MenTrain['Season'] == 2024])
x_women_fit, y_women_fit = x_y_from_data_frame(WomenTrain[WomenTrain['Season'] == 2024])

x_men, y_men = x_y_from_data_frame(MenTest[MenTest['Season'] == 2025])
x_women, y_women = x_y_from_data_frame(WomenTest[WomenTest['Season'] == 2025])


mask = ~np.isnan(x_men).any(axis=1) & ~np.isnan(y_men)
x_men = x_men[mask]
y_men = y_men[mask]

mask = ~np.isnan(x_women).any(axis=1) & ~np.isnan(y_women)
x_women = x_women[mask]
y_women = y_women[mask]

best_lr_all.fit(x_men_fit, y_men_fit)
y_prob_men = best_lr_all.predict_proba(x_men)[:, 1]
best_lr_all.fit(x_women_fit, y_women_fit)
y_prob_women = best_lr_all.predict_proba(x_women)[:, 1]
y_prob_men[y_prob_men > 0.6] = 1
y_prob_men[y_prob_men < 0.2] = 0
y_prob_women[y_prob_women > 0.7] = 1
y_prob_women[y_prob_women < 0.2] = 0

men_score = brier_score_loss(y_men, y_prob_men)
women_score = brier_score_loss(y_women, y_prob_women)


print("Predictions for season ", 2025)


print('LogisticRegressionCV for men trained on men', men_score)
print('LogisticRegressionCV for women trained on women', women_score)
print('LogisticRegressionCV for ALL trained on ALL', (men_score + women_score)/2)

Predictions for season  2025
LogisticRegressionCV for men trained on men 0.13071196449456832
LogisticRegressionCV for women trained on women 0.08878655897577922
LogisticRegressionCV for ALL trained on ALL 0.10974926173517377


In [24]:
y_prob_men

array([0.        , 1.        , 0.35059813, 1.        , 1.        ,
       0.        , 0.28717377, 0.55515569, 0.56015189, 0.29353418,
       0.55296938, 0.56788399, 0.46938067, 1.        , 0.52566336,
       0.        , 1.        , 0.        , 0.3537815 , 0.3184889 ,
       1.        , 0.36713196, 0.43068953, 0.44835072, 1.        ,
       1.        , 0.56467663, 0.37865992, 0.49947609, 1.        ,
       0.        , 0.43371503, 1.        , 1.        , 1.        ,
       0.        , 0.30959146, 0.22590213, 0.        , 1.        ,
       1.        , 0.52621698, 0.59216742, 1.        , 0.58122538,
       0.37654108, 1.        , 0.2043995 , 1.        , 0.24219936,
       0.50285644, 1.        , 1.        , 1.        , 0.41786665,
       0.        , 0.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        ])

In [25]:
y_prob_all = np.concatenate([y_prob_men, y_prob_women])
y_prob_all

array([0.        , 1.        , 0.35059813, 1.        , 1.        ,
       0.        , 0.28717377, 0.55515569, 0.56015189, 0.29353418,
       0.55296938, 0.56788399, 0.46938067, 1.        , 0.52566336,
       0.        , 1.        , 0.        , 0.3537815 , 0.3184889 ,
       1.        , 0.36713196, 0.43068953, 0.44835072, 1.        ,
       1.        , 0.56467663, 0.37865992, 0.49947609, 1.        ,
       0.        , 0.43371503, 1.        , 1.        , 1.        ,
       0.        , 0.30959146, 0.22590213, 0.        , 1.        ,
       1.        , 0.52621698, 0.59216742, 1.        , 0.58122538,
       0.37654108, 1.        , 0.2043995 , 1.        , 0.24219936,
       0.50285644, 1.        , 1.        , 1.        , 0.41786665,
       0.        , 0.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 0.44636067, 1.        ,
       0.        , 0.        , 1.        , 0.50394834, 0.56178678,
       0.        , 0.59068196, 1.        , 0.37629763, 0.41738

In [26]:
# 0.23585023867473887

In [27]:
# MenTest and WomenTest have the same columns and the same
# 'Season', 'T1_TeamID', 'T1_Score', 'T2_TeamID', 'T2_Score', 'location'
# values in the same order, but Men have only daata for men teams and NaNs in other rows.
# The same for WomenTest.
# Test below fills the NaNs with values from the data frame, where values are present.
Test = MenTest.combine_first(WomenTest)

In [28]:
best_params_all["C"]  # 0.23585023867473887

0.23664383186227303

In [29]:
x_test, y_test = x_y_from_data_frame(Test)

In [30]:
x_test = np.nan_to_num(x_test, nan=0)

In [31]:
len(y_prob_women) + len(y_prob_men)

126

In [32]:
WomenTrain.to_csv("WomenTrain.csv")
MenTrain.to_csv("MenTrain.csv")
WomenTest.to_csv("WomenTest.csv")
MenTest.to_csv("MenTest.csv")

In [33]:
# Import potrzebnych bibliotek
import pandas as pd
import numpy as np

# 1. Załaduj plik CSV
df1 = pd.read_csv(join('data', 'finalsolution2025.csv'))

df = df1.copy()

# 3. Sprawdź czy długość tablicy zgadza się z liczbą wierszy w pliku CSV
print(f"Długość tablicy y_prop_all: {len(y_prob_all)}")
print(f"Liczba wierszy w CSV: {len(df)}")

# 4. Zastąp kolumnę 'Pred' nowymi wartościami
df['Pred'] = y_prob_all

# 5. Wyświetl kilka pierwszych wierszy, aby zweryfikować zmiany
print("\nPierwsze 5 wierszy po aktualizacji:")
print(df.head())

# 6. Zapisz zaktualizowany plik CSV
df.to_csv('finalsolution2025_updated.csv', index=False)

print("\nPlik został zapisany jako 'finalsolution2025_updated.csv'")

Długość tablicy y_prop_all: 126
Liczba wierszy w CSV: 126

Pierwsze 5 wierszy po aktualizacji:
               ID      Pred  Spread   Usage   ID1   ID2 TeamName.1  \
0  2025_1103_1112  0.000000     -28  Actual  1103  1112      Akron   
1  2025_1104_1140  1.000000      25  Actual  1104  1140    Alabama   
2  2025_1104_1181  0.350598     -20  Actual  1104  1181    Alabama   
3  2025_1104_1352  1.000000       9  Actual  1104  1352    Alabama   
4  2025_1104_1388  1.000000      14  Actual  1104  1388    Alabama   

      TeamName.2  
0        Arizona  
1            BYU  
2           Duke  
3  Robert Morris  
4   St Mary's CA  

Plik został zapisany jako 'finalsolution2025_updated.csv'


In [34]:
final = df.copy()

In [35]:
final = final[['ID', 'Pred']]


In [36]:
final

,ID,Pred
0,2025_1103_1112,0.000000
1,2025_1104_1140,1.000000
2,2025_1104_1181,0.350598
3,2025_1104_1352,1.000000
4,2025_1104_1388,1.000000
...,...,...
121,2025_3380_3417,0.000000
122,2025_3395_3400,0.341775
123,2025_3397_3400,0.274702
124,2025_3400_3456,1.000000


In [37]:
SampleSubmissionStage2 = pd.read_csv(join('../data', 'SampleSubmissionStage2.csv'))


In [38]:
SampleSubmissionStage2

,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5
...,...,...
131402,2025_3477_3479,0.5
131403,2025_3477_3480,0.5
131404,2025_3478_3479,0.5
131405,2025_3478_3480,0.5


In [39]:
final_df = final
sample_df1 = SampleSubmissionStage2

sample_df = sample_df1.copy()

# Create a mapping between the two dataframes based on ID
# Use left join to keep all rows from sample_df
merged_df = pd.merge(sample_df, final_df, on='ID', how='left', suffixes=('_original', ''))

# Where Pred from final_df is not null, use it, otherwise keep the original Pred
merged_df['Pred_final'] = merged_df['Pred'].fillna(merged_df['Pred_original'])

# Create the updated dataframe with the original columns
updated_df = merged_df[['ID', 'Pred_final']].rename(columns={'Pred_final': 'Pred'})

# Save the updated dataframe
updated_df.to_csv('UpdatedSampleSubmission.csv', index=False)

In [40]:
updated_df

,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5
...,...,...
131402,2025_3477_3479,0.5
131403,2025_3477_3480,0.5
131404,2025_3478_3479,0.5
131405,2025_3478_3480,0.5


In [41]:
MenTest

,Season,T1_TeamID,T1_Score,T2_TeamID,T2_Score,location,T1_Score_mean,T1_FGM,T1_FGM3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Blk,T1_PF,T1_opponent_FGM,T1_opponent_FGM3,T1_opponent_FTM,T1_opponent_FTA,T1_opponent_OR,T1_opponent_DR,T1_opponent_Blk,T1_opponent_PF,T1_PointDiff,T2_Score_mean,T2_FGM,T2_FGM3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Blk,T2_PF,T2_opponent_Score,T2_opponent_FGM,T2_opponent_FGM3,T2_opponent_FTM,T2_opponent_FTA,T2_opponent_OR,T2_opponent_DR,T2_opponent_Blk,T2_opponent_PF,T2_PointDiff,T1_win_ratio_14d,T2_win_ratio_14d,T1_seed,T2_seed,Seed_diff,T1_TeamELO,T2_TeamELO,T1_CoachELO,T2_CoachELO
0,2025.0,1101.0,NaN,1102.0,NaN,NaN,68.333333,26.0,3.333333,13.0,16.666667,12.0,20.333333,4.333333,26.333333,25.666667,3.333333,19.666667,30.666667,8.333333,24.0,7.0,18.333333,-6.0,53.000000,18.500000,8.50,7.50,11.500000,5.000000,20.00,2.00,16.000000,77.500000,27.00,10.50,13.000000,18.50,8.000000,25.500000,2.000000,15.000000,-24.500000,0.333333,0.0,NaN,NaN,NaN,1459.186427,1285.078651,1557.695414,1388.153462
1,2025.0,1101.0,NaN,1103.0,NaN,NaN,68.333333,26.0,3.333333,13.0,16.666667,12.0,20.333333,4.333333,26.333333,25.666667,3.333333,19.666667,30.666667,8.333333,24.0,7.0,18.333333,-6.0,90.000000,31.500000,11.25,15.75,20.750000,11.250000,21.50,4.00,15.500000,75.250000,26.25,8.00,14.750000,18.25,8.750000,19.500000,1.500000,17.750000,14.750000,0.333333,1.0,NaN,13.0,NaN,1459.186427,1727.740773,1557.695414,1856.454628
2,2025.0,1101.0,NaN,1104.0,NaN,NaN,68.333333,26.0,3.333333,13.0,16.666667,12.0,20.333333,4.333333,26.333333,25.666667,3.333333,19.666667,30.666667,8.333333,24.0,7.0,18.333333,-6.0,92.000000,33.250000,9.50,16.00,23.250000,8.500000,27.75,3.25,19.750000,91.000000,32.00,8.75,18.250000,27.25,10.000000,26.500000,4.750000,19.000000,1.000000,0.333333,0.5,NaN,2.0,NaN,1459.186427,2069.064662,1557.695414,2129.457038
3,2025.0,1101.0,NaN,1105.0,NaN,NaN,68.333333,26.0,3.333333,13.0,16.666667,12.0,20.333333,4.333333,26.333333,25.666667,3.333333,19.666667,30.666667,8.333333,24.0,7.0,18.333333,-6.0,54.666667,18.333333,8.00,10.00,14.333333,8.333333,18.00,4.00,16.666667,70.333333,25.00,7.00,13.333333,17.00,6.333333,24.333333,3.333333,16.333333,-15.666667,0.333333,0.0,NaN,NaN,NaN,1459.186427,1025.906452,1557.695414,1271.731597
4,2025.0,1101.0,NaN,1106.0,NaN,NaN,68.333333,26.0,3.333333,13.0,16.666667,12.0,20.333333,4.333333,26.333333,25.666667,3.333333,19.666667,30.666667,8.333333,24.0,7.0,18.333333,-6.0,66.600000,23.400000,9.00,10.80,15.600000,10.800000,24.00,2.60,15.400000,61.800000,21.80,5.60,12.600000,18.20,8.600000,26.000000,3.800000,14.400000,4.800000,0.333333,1.0,NaN,16.0,NaN,1459.186427,1235.938982,1557.695414,1482.278882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131402,2025.0,3477.0,NaN,3479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1113.947230,1190.234158,1612.184052,1612.184052
131403,2025.0,3477.0,NaN,3480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1113.947230,1351.409092,1612.184052,1612.184052
131404,2025.0,3478.0,NaN,3479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1178.688631,1190.234158,1612.184052,1612.184052
131405,2025.0,3478.0,NaN,3480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1178.688631,1351.409092,1612.184052,1612.184052


In [42]:
updated_df[updated_df['Pred']!=0.5]

,ID,Pred
732,2025_1103_1112,0.000000
1116,2025_1104_1140,1.000000
1156,2025_1104_1181,0.350598
1322,2025_1104_1352,1.000000
1356,2025_1104_1388,1.000000
...,...,...
126882,2025_3380_3417,0.000000
128090,2025_3395_3400,0.341775
128249,2025_3397_3400,0.274702
128532,2025_3400_3456,1.000000
